In [ ]:
import pandas as pd
from google.colab import drive
import openpyxl

# 1. Mount Drive
drive.mount('/content/drive')




file_path = '/content/drive/MyDrive/traffic/alltime_combined_traffic_data.csv'

# 3. Read the CSV file
alltime = pd.read_csv(file_path)
file_path = '/content/drive/MyDrive/traffic/nighttime_combined_traffic_data.csv'

# 3. Read the CSV file
nighttime = pd.read_csv(file_path)


In [ ]:
# @title
import json
from urllib.request import urlopen

import pandas as pd
import plotly.express as px


# ---------------------------------------------------------
# 1. Clean your county-level campaign data
# ---------------------------------------------------------
campaign_df = alltime[['County', 'Campaign_Group']].drop_duplicates().reset_index(drop=True)

campaign_df["County"] = (
    campaign_df["County"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+County$", "", regex=True)
)


# ---------------------------------------------------------
# 2. Load US county boundaries
# ---------------------------------------------------------
geojson_url = (
    "https://raw.githubusercontent.com/plotly/datasets/"
    "master/geojson-counties-fips.json"
)

with urlopen(geojson_url) as response:
    us_counties = json.load(response)


# ---------------------------------------------------------
# 3. Keep only Ohio counties
# Ohio's state FIPS code is 39
# ---------------------------------------------------------
ohio_counties = {
    "type": "FeatureCollection",
    "features": [
        feature
        for feature in us_counties["features"]
        if feature["properties"]["STATE"] == "39"
    ],
}


# ---------------------------------------------------------
# 4. Create a County-to-FIPS lookup from the GeoJSON
# ---------------------------------------------------------
ohio_fips = pd.DataFrame(
    {
        "County": [
            feature["properties"]["NAME"]
            for feature in ohio_counties["features"]
        ],
        "fips": [
            feature["id"]
            for feature in ohio_counties["features"]
        ],
    }
)


# Keep every Ohio county, even if campaign data is missing
map_df = ohio_fips.merge(
    campaign_df,
    on="County",
    how="left",
    validate="one_to_one",
)

map_df["Campaign_Group"] = map_df["Campaign_Group"].fillna(
    "Missing / Unassigned"
)


# ---------------------------------------------------------
# 5. Check for county names that did not match
# ---------------------------------------------------------
unmatched = campaign_df.loc[
    ~campaign_df["County"].isin(ohio_fips["County"]),
    "County",
].unique()

if len(unmatched) > 0:
    print("County names that did not match:", unmatched)


# ---------------------------------------------------------
# 6. Plot the campaign groups
# ---------------------------------------------------------
color_map = {
    "Treated (Speeding)": "#E76F51",
    "Treated (Nighttime)": "#457B9D",
    "Control": "#BDBDBD",
    "Missing / Unassigned": "#F2F2F2",
}

fig = px.choropleth(
    map_df,
    geojson=ohio_counties,
    locations="fips",
    color="Campaign_Group",
    hover_name="County",
    hover_data={
        "fips": False,
        "Campaign_Group": True,
    },
    color_discrete_map=color_map,
    category_orders={
        "Campaign_Group": [
            "Treated (Speeding)",
            "Treated (Nighttime)",
            "Control",
            "Missing / Unassigned",
        ]
    },
    title="Ohio Counties by Campaign Group",
)

fig.update_geos(
    fitbounds="locations",
    visible=False,
)

fig.update_traces(
    marker_line_color="white",
    marker_line_width=0.8,
)

fig.update_layout(
    height=700,
    margin={"r": 0, "t": 60, "l": 0, "b": 0},
    legend_title_text="Campaign Group",
)

fig.show()

In [ ]:
!pip install meteostat

In [ ]:
!    pip install "datacommons-client[Pandas]"


In [ ]:
# @title
import numpy as np
import pandas as pd
import meteostat as ms

from datacommons_client.client import DataCommonsClient

client = DataCommonsClient(
    api_key=__import__("os").environ["DATACOMMONS_API_KEY"]
)

In [ ]:
import pandas as pd
import meteostat as ms


# ---------------------------------------------------------
# Get Ohio counties from Data Commons
# ---------------------------------------------------------

response = client.node.fetch_property_values(
    node_dcids=["geoId/39"],
    properties="containedInPlace+{typeOf:County}",
    out=False
).to_dict()

county_nodes = (
    response["data"]["geoId/39"]["arcs"]
    ["containedInPlace+"]["nodes"]
)

county_locations = pd.DataFrame(
    county_nodes
)[["dcid", "name"]]

county_locations["County"] = (
    county_locations["name"]
    .str.replace(" County", "", regex=False)
)


# ---------------------------------------------------------
# Get county coordinates
# ---------------------------------------------------------

coordinate_response = (
    client.node.fetch_property_values(
        node_dcids=county_locations["dcid"].tolist(),
        properties=["latitude", "longitude"]
    )
    .to_dict()["data"]
)


def get_coordinate(dcid, variable):

    nodes = (
        coordinate_response[dcid]["arcs"]
        [variable]["nodes"]
    )

    return float(nodes[0]["value"])


county_locations["Latitude"] = (
    county_locations["dcid"]
    .apply(lambda x: get_coordinate(x, "latitude"))
)

county_locations["Longitude"] = (
    county_locations["dcid"]
    .apply(lambda x: get_coordinate(x, "longitude"))
)


# ---------------------------------------------------------
# Clean dates
# ---------------------------------------------------------

alltime["Date"] = pd.to_datetime(
    alltime["Date"],
    format="mixed"
)

nighttime["Date"] = pd.to_datetime(
    nighttime["Date"],
    format="mixed"
)

start_date = alltime["Date"].min().date()
end_date = alltime["Date"].max().date()


# ---------------------------------------------------------
# Download weather data
# ---------------------------------------------------------

weather_data = []
missing_counties = []

for _, county in county_locations.iterrows():

    point = ms.Point(
        float(county["Latitude"]),
        float(county["Longitude"])
    )

    stations = ms.stations.nearby(
        point,
        radius=150_000,
        limit=4
    )

    if stations is None or len(stations) == 0:

        missing_counties.append(county["County"])
        print(f"No stations found for {county['County']}")

        continue

    daily = ms.daily(
        stations,
        start_date,
        end_date,
        parameters=[
            ms.Parameter.TEMP,
            ms.Parameter.PRCP,
            ms.Parameter.SNWD,
            ms.Parameter.WSPD
        ]
    )

    daily = ms.interpolate(
        daily,
        point
    ).fetch()

    if daily.empty:

        print(f"No weather data found for {county['County']}")
        continue

    daily.index.name = "Date"
    daily = daily.reset_index()

    daily.columns = [
        getattr(column, "value", column)
        for column in daily.columns
    ]

    daily["County"] = county["County"]

    weather_data.append(daily)


print("Counties without stations:", missing_counties)

if len(weather_data) == 0:
    raise ValueError("No weather data were downloaded.")


# ---------------------------------------------------------
# Combine daily weather
# ---------------------------------------------------------

weather_daily = pd.concat(
    weather_data,
    ignore_index=True
)

weather_daily["Date"] = pd.to_datetime(
    weather_daily["Date"]
)


# ---------------------------------------------------------
# Aggregate daily weather to weekly weather
# ---------------------------------------------------------

weather_weekly = (
    weather_daily
    .groupby(
        [
            "County",
            pd.Grouper(
                key="Date",
                freq="W-SUN"
            )
        ],
        as_index=False
    )
    .agg(
        temp_c=("temp", "mean"),
        precipitation_mm=("prcp", "sum"),
        snow_depth_cm=("snwd", "max"),
        wind_speed_kph=("wspd", "mean")
    )
)


# ---------------------------------------------------------
# Merge weather into both datasets
# ---------------------------------------------------------

alltime_cov = alltime.merge(
    weather_weekly,
    on=["County", "Date"],
    how="left",
    validate="many_to_one"
)

nighttime_cov = nighttime.merge(
    weather_weekly,
    on=["County", "Date"],
    how="left",
    validate="many_to_one"
)


# Check the merge
display(
    alltime_cov[
        [
            "County",
            "Date",
            "temp_c",
            "precipitation_mm",
            "snow_depth_cm",
            "wind_speed_kph"
        ]
    ].head()
)

In [ ]:
demographic_variables = {
    "Median_Income_Household":
        "median_household_income",

    "UnemploymentRate_Person":
        "unemployment_rate",

    "Count_Person":
        "population",

    "Count_Person_18To24Years":
        "population_18_24",

    "Count_Person_65OrMoreYears":
        "population_65_plus",

    "Count_Person_25OrMoreYears":
        "population_25_plus",

    "Count_Person_25OrMoreYears_EducationalAttainmentBachelorsDegreeOrHigher":
        "population_bachelors_plus"
}


demographics_long = client.observations_dataframe(
    variable_dcids=list(demographic_variables.keys()),
    date="2024",
    entity_dcids=county_locations["dcid"].tolist(),
    property_filters={
        "measurementMethod": "CensusACS5yrSurvey"
    }
)


demographics = (
    demographics_long
    .pivot_table(
        index="entity",
        columns="variable",
        values="value",
        aggfunc="first"
    )
    .reindex(columns=demographic_variables.keys())
    .rename(columns=demographic_variables)
    .reset_index()
    .rename(columns={"entity": "dcid"})
)


demographics = county_locations[
    ["dcid", "County"]
].merge(
    demographics,
    on="dcid",
    how="left"
)


# Create model-ready variables
demographics["log_median_household_income"] = np.log(
    demographics["median_household_income"]
)

demographics["pct_age_18_24"] = (
    100
    * demographics["population_18_24"]
    / demographics["population"]
)

demographics["pct_age_65_plus"] = (
    100
    * demographics["population_65_plus"]
    / demographics["population"]
)

demographics["pct_bachelors_plus"] = (
    100
    * demographics["population_bachelors_plus"]
    / demographics["population_25_plus"]
)


demographic_columns = [
    "County",
    "log_median_household_income",
    "unemployment_rate",
    "pct_age_18_24",
    "pct_age_65_plus",
    "pct_bachelors_plus"
]


alltime_cov = alltime_cov.merge(
    demographics[demographic_columns],
    on="County",
    how="left",
    validate="many_to_one"
)

nighttime_cov = nighttime_cov.merge(
    demographics[demographic_columns],
    on="County",
    how="left",
    validate="many_to_one"
)


# Check missing values
display(
    demographics[demographic_columns]
    .isna()
    .sum()
)

In [ ]:


alltime_cov = alltime_cov.drop(columns=['snow_depth_cm','unemployment_rate', 'pct_bachelors_plus' ])
nighttime_cov = nighttime_cov.drop(columns=['snow_depth_cm','unemployment_rate', 'pct_bachelors_plus' ])

In [ ]:
# @title
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pandas.api.types import is_numeric_dtype


common_columns = alltime_cov.columns.intersection(nighttime_cov.columns)

for col in common_columns:

    # -----------------------------------------------------
    # Date columns: display date ranges side by side
    # -----------------------------------------------------
    if col.lower() == "date":

        alltime_dates = pd.to_datetime(
            alltime_cov[col],
            errors="coerce"
        )

        nighttime_dates = pd.to_datetime(
            nighttime_cov[col],
            errors="coerce"
        )

        date_summary = pd.DataFrame({
            "Dataset": ["All Time", "Nighttime"],
            "Minimum Date": [
                alltime_dates.min(),
                nighttime_dates.min()
            ],
            "Maximum Date": [
                alltime_dates.max(),
                nighttime_dates.max()
            ],
            "Missing": [
                alltime_dates.isna().sum(),
                nighttime_dates.isna().sum()
            ]
        })

        display(date_summary)
        continue

    # Determine whether the column should be continuous
    both_numeric = (
        is_numeric_dtype(alltime_cov[col])
        and is_numeric_dtype(nighttime_cov[col])
    )

    combined_values = pd.concat(
        [alltime_cov[col], nighttime_cov[col]],
        ignore_index=True
    )

    is_continuous = (
        both_numeric
        and combined_values.nunique(dropna=True) > 10
    )

    # -----------------------------------------------------
    # Continuous variables
    # -----------------------------------------------------
    if is_continuous:

        alltime_values = alltime_cov[col].dropna()
        nighttime_values = nighttime_cov[col].dropna()

        # Use the same bins for both graphs
        combined_nonmissing = pd.concat(
            [alltime_values, nighttime_values],
            ignore_index=True
        )

        bin_edges = np.histogram_bin_edges(
            combined_nonmissing,
            bins="auto"
        )

        with sns.axes_style("white"):
            fig, axs = plt.subplots(
                nrows=1,
                ncols=2,
                figsize=(15, 5),
                sharex=True,
                sharey=True
            )

            sns.histplot(
                alltime_values,
                bins=bin_edges,
                stat="density",
                kde=True,
                color="#66C2A5",
                alpha=0.4,
                ax=axs[0]
            )

            sns.histplot(
                nighttime_values,
                bins=bin_edges,
                stat="density",
                kde=True,
                color="#FC8D62",
                alpha=0.4,
                ax=axs[1]
            )

            axs[0].set_title(
                f"All Time: {col}\n"
                f"n={len(alltime_values):,}, "
                f"missing={alltime_cov[col].isna().sum():,}"
            )

            axs[1].set_title(
                f"Nighttime: {col}\n"
                f"n={len(nighttime_values):,}, "
                f"missing={nighttime_cov[col].isna().sum():,}"
            )

            axs[0].set_xlabel(col)
            axs[1].set_xlabel(col)
            axs[0].set_ylabel("Density")
            axs[1].set_ylabel("Density")

            fig.suptitle(
                f"Distribution of {col}: All Time vs. Nighttime",
                fontsize=14
            )

            plt.tight_layout()
            plt.show()

    # -----------------------------------------------------
    # Categorical variables
    # -----------------------------------------------------
    else:

        alltime_values = (
            alltime_cov[col]
            .fillna("Missing")
            .astype(str)
        )

        nighttime_values = (
            nighttime_cov[col]
            .fillna("Missing")
            .astype(str)
        )

        # Ensure that both plots show categories in the same order
        category_order = (
            pd.concat(
                [alltime_values, nighttime_values]
            )
            .value_counts()
            .index
            .tolist()
        )



        # Also display aligned counts
        count_table = pd.concat(
            [
                alltime_values.value_counts().rename("All Time"),
                nighttime_values.value_counts().rename("Nighttime")
            ],
            axis=1
        ).fillna(0).astype(int)

        display(count_table)
print(alltime_cov.isna().sum())

In [ ]:
# @title
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns


campaign_order = [
    "Control",
    "Treated (Speeding)",
    "Treated (Nighttime)"
]

campaign_colors = {
    "Control": "#7F7F7F",
    "Treated (Speeding)": "#E76F51",
    "Treated (Nighttime)": "#457B9D"
}

statistics = [
    ("Average", "Average across counties"),
    ("Median", "Median across counties"),
    ("Minimum", "Minimum across counties"),
    ("Maximum", "Maximum across counties")
]


# ---------------------------------------------------------
# Create daily/weekly time-series statistics
# ---------------------------------------------------------
def create_time_series_metrics(df):

    time_series = df.copy()

    time_series["Date"] = pd.to_datetime(
        time_series["Date"],
        errors="coerce"
    )

    time_series["Value"] = pd.to_numeric(
        time_series["Value"],
        errors="coerce"
    )

    time_series["Metric"] = (
        time_series["Metric"]
        .astype(str)
        .str.strip()
        .str.title()
    )

    time_series["Campaign_Group"] = (
        time_series["Campaign_Group"]
        .astype(str)
        .str.strip()
    )

    time_series["Period"] = (
        time_series["Period"]
        .astype(str)
        .str.strip()
        .str.title()
    )

    time_series = (
        time_series
        .groupby(
            [
                "Date",
                "Metric",
                "Campaign_Group",
                "Period"
            ],
            observed=True,
            as_index=False
        )
        .agg(
            Average=("Value", "mean"),
            Median=("Value", "median"),
            Minimum=("Value", "min"),
            Maximum=("Value", "max"),
            Standard_Deviation=("Value", "std"),
            County_Count=("County", "nunique")
        )
    )

    return time_series
alltime_time_series = create_time_series_metrics(alltime)
nighttime_time_series = create_time_series_metrics(nighttime)
def plot_time_series(time_series_df, dataset_name):

    metrics = (
        time_series_df["Metric"]
        .dropna()
        .drop_duplicates()
        .tolist()
    )

    post_start = time_series_df.loc[
        time_series_df["Period"] == "Post",
        "Date"
    ].min()

    date_min = time_series_df["Date"].min()
    date_max = time_series_df["Date"].max()

    sns.set_theme(style="whitegrid")

    for metric in metrics:


        metric_data = time_series_df.loc[
            time_series_df["Metric"] == metric
        ].copy()

        fig, axs = plt.subplots(
            nrows=2,
            ncols=2,
            figsize=(18, 11),
            sharex=True
        )

        axs = axs.flatten()

        for ax, (statistic, statistic_label) in zip(
            axs,
            statistics
        ):

            # Pre-period shading
            ax.axvspan(
                date_min,
                post_start,
                color="#D9D9D9",
                alpha=0.18
            )

            # Post-period shading
            ax.axvspan(
                post_start,
                date_max,
                color="#F6E8A6",
                alpha=0.18
            )

            for campaign_group in campaign_order:

                group_data = (
                    metric_data.loc[
                        metric_data["Campaign_Group"]
                        == campaign_group
                    ]
                    .sort_values("Date")
                )

                ax.plot(
                    group_data["Date"],
                    group_data[statistic],
                    color=campaign_colors[campaign_group],
                    linewidth=2,
                    marker="o",
                    markersize=3,
                    label=campaign_group
                )

            # Campaign start
            ax.axvline(
                post_start,
                color="black",
                linestyle="--",
                linewidth=1.5
            )

            ax.set_title(statistic_label)
            ax.set_xlabel("Date")
            ax.set_ylabel(statistic)

            ax.xaxis.set_major_locator(
                mdates.MonthLocator(interval=2)
            )

            ax.xaxis.set_major_formatter(
                mdates.DateFormatter("%b\n%Y")
            )

            ax.text(
                0.02,
                0.95,
                "Pre",
                transform=ax.transAxes,
                ha="left",
                va="top",
                color="#555555"
            )

            ax.text(
                0.98,
                0.95,
                "Post",
                transform=ax.transAxes,
                ha="right",
                va="top",
                color="#555555"
            )

            ax.grid(alpha=0.25)

        # Create one shared legend
        handles, labels = axs[0].get_legend_handles_labels()

        fig.legend(
            handles,
            labels,
            loc="upper center",
            bbox_to_anchor=(0.5, 0.95),
            ncol=3,
            frameon=False
        )

        fig.suptitle(
            f"{dataset_name}: {metric} Over Time",
            fontsize=16,
            y=1.00
        )

        plt.tight_layout(
            rect=[0, 0, 1, 0.91]
        )

        plt.show()
        plt.close(fig)
plot_time_series(
    alltime_time_series,
    "All Time"
)

plot_time_series(
    nighttime_time_series,
    "Nighttime"
)



The logged results support parallel trends much more clearly. None of the treatment-by-trend interactions are statistically significant

In [ ]:
# @title
import numpy as np

metrics = alltime["Metric"].unique()

alltime["Date"] = pd.to_datetime(
    alltime["Date"],
    format="mixed"
)

alltime["Value"] = pd.to_numeric(
    alltime["Value"],
    errors="coerce"
)

alltime["log_Value"] = np.log1p(
    alltime["Value"]
)

event_date = alltime.loc[
    alltime["Period"] == "Post",
    "Date"
].min()

alltime["trend"] = (
    alltime["Date"] - event_date
).dt.days // 7


for metric in metrics:

    print(metric, "metric")

    model_df = alltime.loc[
        (alltime["Post"] == 0)
        & (alltime["Metric"] == metric)
    ].dropna(
        subset=[
            "Value",
            "Campaign_Group",
            "trend",
            "Group"
        ]
    ).copy()

    formula = (
        "Value"
        + " ~ C(Campaign_Group)"
        + " + trend"
        + " + C(Campaign_Group):trend"
    )

    trend_fit = smf.ols(
        formula,
        data=model_df
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups": model_df["Group"],
            "use_correction": True,
            "df_correction": True
        },
        use_t=True
    )

    print(trend_fit.summary())

# add more covariates
#summarize results and next steps more closely
#deal with multiple treatments
# comopare aic and bic

In [ ]:
# @title
import numpy as np
import statsmodels.formula.api as smf
from scipy.stats import chi2

metrics = alltime["Metric"].unique()

alltime["Date"] = pd.to_datetime(
    alltime["Date"],
    format="mixed"
)

alltime["Value"] = pd.to_numeric(
    alltime["Value"],
    errors="coerce"
)

alltime["log_Value"] = np.log1p(
    alltime["Value"]
)

event_date = alltime.loc[
    alltime["Period"] == "Post",
    "Date"
].min()

alltime["trend"] = (
    alltime["Date"] - event_date
).dt.days // 7


for metric in metrics:

    print(metric, "metric")

    model_df = alltime.loc[
        (alltime["Post"] == 0)
        & (alltime["Metric"] == metric)
    ].dropna(
        subset=[
            "log_Value",
            "Campaign_Group",
            "trend",
            "Group"
        ]
    ).copy()

    formula = (
        "log_Value"
        + " ~ C(Campaign_Group)"
        + " + trend"
        + " + C(Campaign_Group):trend"
    )

    trend_fit = smf.ols(
        formula,
        data=model_df
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups": model_df["Group"],
            "use_correction": True,
            "df_correction": True
        },
        use_t=True
    )

    print(trend_fit.summary())

# add more covariates
#summarize results and next steps more closely
#deal with multiple treatments
# comopare aic and bic

In [ ]:
%pip install meteostat "datacommons-client[Pandas]"

for excessive speeding across the speeding treatment, there seems to be a

In [ ]:
# @title
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


reduced_metrics = [
    "Average",
    "Median",
    "Minimum",
    "Maximum"
]


def plot_metric_distributions(
    county_metrics,
    dataset_name
):

    original_metrics = (
        county_metrics["Metric"]
        .dropna()
        .drop_duplicates()
        .tolist()
    )

    for original_metric in original_metrics:

        # Data for one original metric,
        # such as Hard Acceleration
        full_df = county_metrics.loc[
            county_metrics["Metric"] == original_metric
        ]

        for metric in reduced_metrics:

            values = full_df[metric].dropna()

            if values.empty:
                continue

            if (values < 0).any():
                print(
                    f"Skipping log transform for "
                    f"{original_metric} — {metric}: "
                    "negative values found."
                )
                continue

            with sns.axes_style("white"):

                fig, axs = plt.subplots(
                    ncols=2,
                    figsize=(15, 5)
                )

                # Original distribution
                sns.histplot(
                    values,
                    ax=axs[0],
                    kde=True
                )

                # Log-transformed distribution
                sns.histplot(
                    np.log1p(values),
                    ax=axs[1],
                    kde=True
                )

                axs[0].set_xlabel(metric)

                axs[1].set_xlabel(
                    f"log({metric} + 1)"
                )

                axs[0].set_title(
                    f"Original {metric}"
                )

                axs[1].set_title(
                    f"Log-Transformed {metric}"
                )

                fig.suptitle(
                    f"{dataset_name}: {original_metric}",
                    fontsize=14
                )

                fig.subplots_adjust(
                    wspace=0.5,
                    top=0.82
                )

                plt.show()
                plt.close(fig)
def create_county_metrics(df):

    metric_df = df.copy()

    metric_df["Value"] = pd.to_numeric(
        metric_df["Value"],
        errors="coerce"
    )

    metric_df["Period"] = (
        metric_df["Period"]
        .astype(str)
        .str.strip()
        .str.title()
    )

    metric_df["Metric"] = (
        metric_df["Metric"]
        .astype(str)
        .str.strip()
        .str.title()
    )

    metric_df["Campaign_Group"] = (
        metric_df["Campaign_Group"]
        .astype(str)
        .str.strip()
    )

    county_metrics = (
        metric_df
        .groupby(
            [
                "County",
                "Campaign_Group",
                "Period",
                "Metric"
            ],
            observed=True
        )
        .agg(
            Average=("Value", "mean"),
            Median=("Value", "median"),
            Minimum=("Value", "min"),
            Maximum=("Value", "max"),
            Standard_Deviation=("Value", "std"),
            Observations=("Value", "count")
        )
        .reset_index()
    )

    # Match the notebook's log(x + 1) transformation
    for statistic in [
        "Average",
        "Median",
        "Minimum",
        "Maximum"
    ]:

        if (county_metrics[statistic].dropna() < 0).any():
            raise ValueError(
                f"{statistic} contains negative values, so "
                "log(x + 1) may not be appropriate."
            )

        county_metrics[f"Log_{statistic}"] = np.log1p(
            county_metrics[statistic]
        )

    return county_metrics


alltime_county_metrics = create_county_metrics(alltime_cov)
nighttime_county_metrics = create_county_metrics(nighttime_cov)
plot_metric_distributions(
    alltime_county_metrics,
    "All Time"
)

plot_metric_distributions(
    nighttime_county_metrics,
    "Nighttime"
)

In [ ]:
alltime_cov
id_cols = [
        "Date", "County", "Group", "Population",
        "Treated", "Campaign_Group", "Post",
        "Period", "Storm_Week", "Week_Rel", "temp_c",
        "precipitation_mm", "wind_speed_kph", "log_median_household_income", "pct_age_18_24", "pct_age_65_plus"
    ],
    wide = (
        df.pivot_table(
            index=id_cols,
            columns="Metric",
            values="Value",
            aggfunc="first"
        )
        .reset_index()
    )

In [ ]:
# @title
def make_wide(df):

    df = df.copy()

    # Make metric names usable as column names
    df["Metric"] = (
        df["Metric"]
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

    id_cols = [
        "Date", "County", "Group", "Population",
        "Treated", "Campaign_Group", "Post",
        "Period", "Storm_Week", "Week_Rel", "temp_c",
        "precipitation_mm", "wind_speed_kph", "log_median_household_income", "pct_age_18_24", "pct_age_65_plus"
    ]

    wide = (
        df.pivot_table(
            index=id_cols,
            columns="Metric",
            values="Value",
            aggfunc="first"
        )
        .reset_index()
    )

    wide.columns.name = None

    # Add log versions of every metric
    metric_cols = df["Metric"].unique()

    for col in metric_cols :
        wide[f"log_{col}"] = np.log1p(wide[col])
    for col in ['Population', "Storm_Week"]:
        wide[f"log_{col}"] = np.log1p(wide[col])
    for col in ["pct_age_18_24", "pct_age_65_plus","temp_c",
        "precipitation_mm", "wind_speed_kph"]:
        wide[f"log_{col}"] = np.log1p(wide[col])
    return wide
alltime_wide = make_wide(alltime_cov)
nighttime_wide = make_wide(nighttime_cov)

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


metrics = [i for i in list(set(alltime_wide.columns.append(nighttime_wide.columns))) if i not in [
        "Date", "County", "Group",
        "Treated", "Campaign_Group", "Post",
        "Period",  "Week_Rel"
    ]]
metrics =  metrics + ["temp_c",
        "precipitation_mm", "wind_speed_kph", "log_median_household_income", "pct_age_18_24", "pct_age_65_plus"]
def plot_metric_distributions(df, dataset_name):

    for metric in metrics:
        print(metric, "metric")

        log_metric = f"log_{metric}"

        if (
            metric not in df.columns
            or log_metric not in df.columns
        ):
            continue

        with sns.axes_style("white"):

            fig, axs = plt.subplots(
                ncols=2,
                figsize=(15, 5)
            )

            sns.histplot(
                df[metric].dropna(),
                kde=True,
                ax=axs[0]
            )

            sns.histplot(
                df[log_metric].dropna(),
                kde=True,
                ax=axs[1]
            )

            axs[0].set_title(f"Original: {metric}")
            axs[1].set_title(f"Log: {metric}")

            axs[0].set_xlabel(metric)
            axs[1].set_xlabel(log_metric)

            fig.suptitle(
                f"{dataset_name}: {metric}",
                fontsize=14
            )

            plt.tight_layout()
            plt.show()
            plt.close(fig)



plot_metric_distributions(
    alltime_wide,
    "All Time"
)

plot_metric_distributions(
    nighttime_wide,
    "Nighttime"
)


In [ ]:
# @title
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


def plot_mixed_diagnostics(mm_fit, model_df, metric):

    # Rows actually used by Statsmodels
    used_rows = mm_fit.model.data.row_labels
    diagnostic_df = model_df.loc[used_rows].copy().reset_index(drop=True)

    y = np.asarray(mm_fit.model.endog)

    # Marginal fitted values: fixed effects only, X beta
    marginal_fitted = (
        mm_fit.model.exog @ np.asarray(mm_fit.fe_params)
    )

    marginal_residuals = y - marginal_fitted

    # Conditional fitted values: fixed + predicted random effects
    conditional_fitted = np.asarray(mm_fit.fittedvalues)
    conditional_residuals = np.asarray(mm_fit.resid)

    standardized_residuals = (
        conditional_residuals / np.sqrt(mm_fit.scale)
    )

    diagnostic_df["Marginal"] = marginal_residuals
    diagnostic_df["Conditional"] = conditional_residuals

    fig, axs = plt.subplots(
        nrows=2,
        ncols=3,
        figsize=(18, 10)
    )

    # 1. Marginal residuals versus fixed-effects fitted values
    sns.regplot(
        x=marginal_fitted,
        y=marginal_residuals,
        lowess=True,
        scatter_kws={"alpha": 0.25},
        line_kws={"color": "red"},
        ax=axs[0, 0]
    )

    axs[0, 0].axhline(0, color="black", linestyle="--")
    axs[0, 0].set_title("Marginal residuals")
    axs[0, 0].set_xlabel("Fixed-effects fitted values")
    axs[0, 0].set_ylabel("Marginal residual")

    # 2. Conditional residuals versus full fitted values
    sns.regplot(
        x=conditional_fitted,
        y=conditional_residuals,
        lowess=True,
        scatter_kws={"alpha": 0.25},
        line_kws={"color": "red"},
        ax=axs[0, 1]
    )

    axs[0, 1].axhline(0, color="black", linestyle="--")
    axs[0, 1].set_title("Conditional residuals")
    axs[0, 1].set_xlabel("Conditional fitted values")
    axs[0, 1].set_ylabel("Conditional residual")

    # 3. Normal Q-Q plot for conditional residuals
    stats.probplot(
        conditional_residuals,
        dist="norm",
        plot=axs[0, 2]
    )

    axs[0, 2].set_title("Q-Q plot: conditional residuals")

    # 4. Marginal and conditional residuals by Group
    group_residuals = diagnostic_df[
        ["Group", "Marginal", "Conditional"]
    ].melt(
        id_vars="Group",
        var_name="Residual type",
        value_name="Residual"
    )

    sns.boxplot(
        data=group_residuals,
        x="Group",
        y="Residual",
        hue="Residual type",
        showfliers=False,
        ax=axs[1, 0]
    )

    axs[1, 0].axhline(0, color="black", linestyle="--")
    axs[1, 0].set_title("Residuals by randomized Group")
    axs[1, 0].tick_params(axis="x", rotation=45)

    # 5. Standardized conditional residuals
    axs[1, 1].scatter(
        np.arange(len(standardized_residuals)),
        standardized_residuals,
        alpha=0.3
    )

    axs[1, 1].axhline(0, color="black", linestyle="--")
    axs[1, 1].axhline(3, color="red", linestyle="--")
    axs[1, 1].axhline(-3, color="red", linestyle="--")
    axs[1, 1].set_title("Standardized conditional residuals")
    axs[1, 1].set_xlabel("Observation")
    axs[1, 1].set_ylabel("Standardized residual")

    # 6. Normal Q-Q plot for Group random effects
    try:
        random_effect_values = []

        for random_effect in mm_fit.random_effects.values():

            names = pd.Index(
                random_effect.index
            ).astype(str)

            # Extract the re1 = Group variance component
            keep = names.str.startswith("re1")

            random_effect_values.extend(
                np.asarray(random_effect)[keep]
            )

        stats.probplot(
            random_effect_values,
            dist="norm",
            plot=axs[1, 2]
        )

        axs[1, 2].set_title("Q-Q plot: Group random effects")

    except Exception as error:
        axs[1, 2].axis("off")
        axs[1, 2].text(
            0.5,
            0.5,
            f"Random effects unavailable:\n{error}",
            ha="center",
            va="center"
        )

    fig.suptitle(
        f"Mixed-model diagnostics: {metric}",
        fontsize=15
    )

    plt.tight_layout()
    plt.show()

    return diagnostic_df

In [ ]:
# @title
from IPython.display import display, Markdown, Math


display(Markdown("## Mixed-Effects Difference-in-Differences Models"))

display(Markdown(
    "For county $c$, randomized group $g$, and relative week $r$, "
    "define the common fixed-effects component as:"
))

display(Math(r"""
\begin{aligned}
\mu_{cgr}
={}& \beta_0
+ \beta_1 \mathrm{Nighttime}_g
+ \beta_2 \mathrm{Speeding}_g
+ \beta_3 \mathrm{Post}_r \\
&+ \beta_4 \log(\mathrm{Population}_c)
+ \beta_5 \mathrm{StormWeek}_r \\
&+ \beta_6(
    \mathrm{Nighttime}_g \times \mathrm{Post}_r
) \\
&+ \beta_7(
    \mathrm{Speeding}_g \times \mathrm{Post}_r
).
\end{aligned}
"""))

display(Markdown("The modeled outcome is:"))

display(Math(r"""
Y_{cgr}=\log(\mathrm{Value}_{cgr}+1)
"""))

display(Markdown(
    "Control and the pre-period are the reference categories."
))


# Model 1
display(Markdown("### 1. Group Random Intercept"))

display(Math(r"""
Y_{cgr}
=
\mu_{cgr}
+
u_g
+
\epsilon_{cgr}
"""))

display(Math(r"""
u_g \sim N(0,\sigma^2_{\mathrm{Group}}),
\qquad
\epsilon_{cgr}\sim N(0,\sigma^2)
"""))

display(Markdown(
    "Observations belonging to the same randomized Group "
    "share a common baseline deviation."
))


# Model 2
display(Markdown("### 2. County Random Intercept"))

display(Math(r"""
Y_{cgr}
=
\mu_{cgr}
+
v_c
+
\epsilon_{cgr}
"""))

display(Math(r"""
v_c \sim N(0,\sigma^2_{\mathrm{County}}),
\qquad
\epsilon_{cgr}\sim N(0,\sigma^2)
"""))

display(Markdown(
    "This model accounts for repeated observations from the same county."
))


# Model 3
display(Markdown("### 3. Group and County Random Intercepts"))

display(Math(r"""
Y_{cgr}
=
\mu_{cgr}
+
u_g
+
v_c
+
\epsilon_{cgr}
"""))

display(Math(r"""
u_g \sim N(0,\sigma^2_{\mathrm{Group}}),
\qquad
v_c \sim N(0,\sigma^2_{\mathrm{County}}),
\qquad
\epsilon_{cgr}\sim N(0,\sigma^2)
"""))

display(Markdown(
    "This model accounts for both randomized Group clustering "
    "and repeated County-level measurements."
))


# Model 4
display(Markdown("### 4. Group and Relative-Week Random Intercepts"))

display(Math(r"""
Y_{cgr}
=
\mu_{cgr}
+
u_g
+
w_r
+
\epsilon_{cgr}
"""))

display(Math(r"""
u_g \sim N(0,\sigma^2_{\mathrm{Group}}),
\qquad
w_r \sim N(0,\sigma^2_{\mathrm{WeekRel}}),
\qquad
\epsilon_{cgr}\sim N(0,\sigma^2)
"""))

display(Markdown(
    "This model accounts for randomized Group clustering and "
    "deviations shared by observations at the same relative week."
))


display(Markdown("### Difference-in-Differences Effects"))

display(Math(r"""
\beta_6
=
\text{Nighttime DiD effect relative to Control}
"""))

display(Math(r"""
\beta_7
=
\text{Speeding DiD effect relative to Control}
"""))

In [ ]:
alltime_wide.columns

In [ ]:
import statsmodels.formula.api as smf
from scipy.stats import chi2
# Refer to this: https://stackoverflow.com/questions/50052421/mixed-models-with-two-random-effects-statsmodels
for metric in log_metrics:
    formula = metric + ' ~ C(Campaign_Group) + C(Post) + log_Population+ C(Storm_Week) + C(Campaign_Group):C(Post)'
    formula0 = metric + ' ~ C(Campaign_Group) + C(Post) + log_Population + C(Storm_Week)'

    vcf = {'re1': '0 + C(County)'}
    alltime_wide['group'] = 1 # All in the same group

    mm = smf.mixedlm(formula, alltime_wide.dropna(subset=[metric]), groups = alltime_wide.dropna(subset=[metric])['group'],
                    vc_formula=vcf, re_formula="0")
    mm0 = smf.mixedlm(formula0, alltime_wide.dropna(subset=[metric]), groups = alltime_wide.dropna(subset=[metric])['group'],
                     vc_formula=vcf, re_formula="0")

    mm_fit = mm.fit(reml=False)
    mm_fit0 = mm0.fit(reml=False)

    p_val = 1 - chi2.cdf(-2*(mm_fit0.llf-mm_fit.llf), df=1)
    lr_stat = 2 * (
        mm_fit.llf - mm_fit0.llf
    )

    p_val = chi2.sf(
        lr_stat,
        df=1
    )

    trt_1_fitted_lift = np.exp(mm_fit.params['C(Campaign_Group)[T.Treated (Nighttime)]:C(Post)[T.1]']) - 1
    trt_1_fitted_lift_se = np.exp(mm_fit.bse['C(Campaign_Group)[T.Treated (Nighttime)]:C(Post)[T.1]']) - 1
    trt_1_fitted_lift_p = mm_fit.pvalues['C(Campaign_Group)[T.Treated (Nighttime)]:C(Post)[T.1]']
    trt_2_fitted_lift = np.exp(mm_fit.params['C(Campaign_Group)[T.Treated (Speeding)]:C(Post)[T.1]']) - 1
    trt_2_fitted_lift_se = np.exp(mm_fit.bse['C(Campaign_Group)[T.Treated (Speeding)]:C(Post)[T.1]']) - 1
    trt_2_fitted_lift_p = mm_fit.pvalues['C(Campaign_Group)[T.Treated (Speeding)]:C(Post)[T.1]']
    print('In {}, including ILS and non-ILS days {} to {}, treatment 1 saw a {}% (+/- {}%) impact relative to control. p-value: {} ' \
    .format(metric, 'pre_start_date', 'post_end_date', round(trt_1_fitted_lift * 100, 2), round(trt_1_fitted_lift_se * 100, 2), round(trt_1_fitted_lift_p, 4)))
    print('In {}, including ILS and non-ILS days {} to {}, treatment 2 saw a {}% (+/- {}%) impact relative to control. p-value: {} ' \
    .format(metric, 'pre_start_date', 'post_end_date', round(trt_2_fitted_lift * 100, 2), round(trt_2_fitted_lift_se * 100, 2), round(trt_2_fitted_lift_p, 4)))
    diagnostic_df = plot_mixed_diagnostics(
      mm_fit,
      alltime_wide.dropna(subset=[metric]),
      metric
    )

In [ ]:
log_metrics = ['log_percentile_speed',
 'log_phone_motion',
 'log_phone_tapping',
 'log_excessive_speeding',
 'log_speed_range',
 'log_hard_acceleration',
 'log_hard_cornering',
 'log_hard_braking',
 'log_handheld_phone_calls']

In [ ]:
import statsmodels.formula.api as smf
from scipy.stats import chi2
# Refer to this: https://stackoverflow.com/questions/50052421/mixed-models-with-two-random-effects-statsmodels
for metric in log_metrics:
    formula = metric + ' ~ C(Campaign_Group) + C(Post) + log_Population+ C(Storm_Week) + C(Campaign_Group):C(Post)'
    formula0 = metric + ' ~ C(Campaign_Group) + C(Post) + log_Population + C(Storm_Week)'

    vcf = {'re1': '0 + C(County)'}
    alltime_wide['group'] = 1 # All in the same group

    mm = smf.mixedlm(formula, alltime_wide.dropna(subset=[metric]), groups = alltime_wide.dropna(subset=[metric])['group'],
                    vc_formula=vcf, re_formula="0")
    mm0 = smf.mixedlm(formula0, alltime_wide.dropna(subset=[metric]), groups = alltime_wide.dropna(subset=[metric])['group'],
                     vc_formula=vcf, re_formula="0")

    mm_fit = mm.fit(reml=False)
    mm_fit0 = mm0.fit(reml=False)

    p_val = 1 - chi2.cdf(-2*(mm_fit0.llf-mm_fit.llf), df=1)
    lr_stat = 2 * (
        mm_fit.llf - mm_fit0.llf
    )

    p_val = chi2.sf(
        lr_stat,
        df=1
    )

    trt_1_fitted_lift = np.exp(mm_fit.params['C(Campaign_Group)[T.Treated (Nighttime)]:C(Post)[T.1]']) - 1
    trt_1_fitted_lift_se = np.exp(mm_fit.bse['C(Campaign_Group)[T.Treated (Nighttime)]:C(Post)[T.1]']) - 1
    trt_1_fitted_lift_p = mm_fit.pvalues['C(Campaign_Group)[T.Treated (Nighttime)]:C(Post)[T.1]']
    trt_2_fitted_lift = np.exp(mm_fit.params['C(Campaign_Group)[T.Treated (Speeding)]:C(Post)[T.1]']) - 1
    trt_2_fitted_lift_se = np.exp(mm_fit.bse['C(Campaign_Group)[T.Treated (Speeding)]:C(Post)[T.1]']) - 1
    trt_2_fitted_lift_p = mm_fit.pvalues['C(Campaign_Group)[T.Treated (Speeding)]:C(Post)[T.1]']
    print('In {}, including ILS and non-ILS days {} to {}, treatment 1 saw a {}% (+/- {}%) impact relative to control. p-value: {} ' \
    .format(metric, 'pre_start_date', 'post_end_date', round(trt_1_fitted_lift * 100, 2), round(trt_1_fitted_lift_se * 100, 2), round(trt_1_fitted_lift_p, 4)))
    print('In {}, including ILS and non-ILS days {} to {}, treatment 2 saw a {}% (+/- {}%) impact relative to control. p-value: {} ' \
    .format(metric, 'pre_start_date', 'post_end_date', round(trt_2_fitted_lift * 100, 2), round(trt_2_fitted_lift_se * 100, 2), round(trt_2_fitted_lift_p, 4)))
    diagnostic_df = plot_mixed_diagnostics(
      mm_fit,
      alltime_wide.dropna(subset=[metric]),
      metric
    )

In [ ]:
import statsmodels.formula.api as smf
from scipy.stats import chi2
# Refer to this: https://stackoverflow.com/questions/50052421/mixed-models-with-two-random-effects-statsmodels

for metric in log_metrics:
    formula = metric + ' ~ C(Campaign_Group) + C(Post) + log_Population+ C(Storm_Week) + temp_c + log_pct_age_18_24+ log_pct_age_65_plus+log_precipitation_mm +log_wind_speed_kph +C(Campaign_Group):C(Post)'
    formula0 = metric + ' ~ C(Campaign_Group) + C(Post) + log_Population + C(Storm_Week)+ temp_c + log_pct_age_18_24+ log_pct_age_65_plus+log_precipitation_mm +log_wind_speed_kph'

    vcf = {'re1': '0 + C(County)'}
    alltime_wide['group'] = 1 # All in the same group

    mm = smf.mixedlm(formula, alltime_wide.dropna(subset=[metric]), groups = alltime_wide.dropna(subset=[metric])['group'],
                    vc_formula=vcf, re_formula="0")
    mm0 = smf.mixedlm(formula0, alltime_wide.dropna(subset=[metric]), groups = alltime_wide.dropna(subset=[metric])['group'],
                     vc_formula=vcf, re_formula="0")

    mm_fit = mm.fit(reml=False)
    mm_fit0 = mm0.fit(reml=False)

    p_val = 1 - chi2.cdf(-2*(mm_fit0.llf-mm_fit.llf), df=1)
    lr_stat = 2 * (
        mm_fit.llf - mm_fit0.llf
    )

    p_val = chi2.sf(
        lr_stat,
        df=1
    )

    trt_1_fitted_lift = np.exp(mm_fit.params['C(Campaign_Group)[T.Treated (Nighttime)]:C(Post)[T.1]']) - 1
    trt_1_fitted_lift_se = np.exp(mm_fit.bse['C(Campaign_Group)[T.Treated (Nighttime)]:C(Post)[T.1]']) - 1
    trt_1_fitted_lift_p = mm_fit.pvalues['C(Campaign_Group)[T.Treated (Nighttime)]:C(Post)[T.1]']
    trt_2_fitted_lift = np.exp(mm_fit.params['C(Campaign_Group)[T.Treated (Speeding)]:C(Post)[T.1]']) - 1
    trt_2_fitted_lift_se = np.exp(mm_fit.bse['C(Campaign_Group)[T.Treated (Speeding)]:C(Post)[T.1]']) - 1
    trt_2_fitted_lift_p = mm_fit.pvalues['C(Campaign_Group)[T.Treated (Speeding)]:C(Post)[T.1]']
    print('In {}, including ILS and non-ILS days {} to {}, treatment 1 saw a {}% (+/- {}%) impact relative to control. p-value: {} ' \
    .format(metric, 'pre_start_date', 'post_end_date', round(trt_1_fitted_lift * 100, 2), round(trt_1_fitted_lift_se * 100, 2), round(trt_1_fitted_lift_p, 4)))
    print('In {}, including ILS and non-ILS days {} to {}, treatment 2 saw a {}% (+/- {}%) impact relative to control. p-value: {} ' \
    .format(metric, 'pre_start_date', 'post_end_date', round(trt_2_fitted_lift * 100, 2), round(trt_2_fitted_lift_se * 100, 2), round(trt_2_fitted_lift_p, 4)))
    diagnostic_df = plot_mixed_diagnostics(
      mm_fit,
      alltime_wide.dropna(subset=[metric]),
      metric
    )

In [ ]:


# Refer to this: https://stackoverflow.com/questions/50052421/mixed-models-with-two-random-effects-statsmodels
log_metrics = [i for i in metrics if 'log' in i and i not in ['log_Storm_Week', 'log_Population']]
for metric in log_metrics:
    formula = metric + ' ~ C(Campaign_Group) + C(Post) + log_Population+ C(Storm_Week) + Week_Rel+ C(Campaign_Group):C(Post)'
    formula0 = metric + ' ~ C(Campaign_Group) + C(Post) + log_Population + C(Storm_Week) + Week_Rel'

    vcf = {'re1': '0 + C(Group)', 're2': '0 + C(County)'}
    alltime_wide['group'] = 1 # All in the same group

    mm = smf.mixedlm(formula, alltime_wide.dropna(subset=[metric]), groups = alltime_wide.dropna(subset=[metric])['group'],
                    vc_formula=vcf, re_formula="0")
    mm0 = smf.mixedlm(formula0, alltime_wide.dropna(subset=[metric]), groups = alltime_wide.dropna(subset=[metric])['group'],
                     vc_formula=vcf, re_formula="0")

    mm_fit = mm.fit(reml=False)
    mm_fit0 = mm0.fit(reml=False)

    p_val = 1 - chi2.cdf(-2*(mm_fit0.llf-mm_fit.llf), df=1)
    lr_stat = 2 * (
        mm_fit.llf - mm_fit0.llf
    )

    p_val = chi2.sf(
        lr_stat,
        df=2
    )

    trt_1_fitted_lift = np.exp(mm_fit.params['C(Campaign_Group)[T.Treated (Nighttime)]:C(Post)[T.1]']) - 1
    trt_1_fitted_lift_se = np.exp(mm_fit.bse['C(Campaign_Group)[T.Treated (Nighttime)]:C(Post)[T.1]']) - 1
    trt_1_fitted_lift_p = mm_fit.pvalues['C(Campaign_Group)[T.Treated (Nighttime)]:C(Post)[T.1]']
    trt_2_fitted_lift = np.exp(mm_fit.params['C(Campaign_Group)[T.Treated (Speeding)]:C(Post)[T.1]']) - 1
    trt_2_fitted_lift_se = np.exp(mm_fit.bse['C(Campaign_Group)[T.Treated (Speeding)]:C(Post)[T.1]']) - 1
    trt_2_fitted_lift_p = mm_fit.pvalues['C(Campaign_Group)[T.Treated (Speeding)]:C(Post)[T.1]']
    print('In {}, including ILS and non-ILS days {} to {}, treatment 1 saw a {}% (+/- {}%) impact relative to control. p-value: {} ' \
    .format(metric, 'pre_start_date', 'post_end_date', round(trt_1_fitted_lift * 100, 2), round(trt_1_fitted_lift_se * 100, 2), round(trt_1_fitted_lift_p, 4)))
    print('In {}, including ILS and non-ILS days {} to {}, treatment 2 saw a {}% (+/- {}%) impact relative to control. p-value: {} ' \
    .format(metric, 'pre_start_date', 'post_end_date', round(trt_2_fitted_lift * 100, 2), round(trt_2_fitted_lift_se * 100, 2), round(trt_2_fitted_lift_p, 4)))
    diagnostic_df = plot_mixed_diagnostics(
      mm_fit,
      alltime_wide.dropna(subset=[metric]),
      metric
    )

In [ ]:
# Refer to this: https://stackoverflow.com/questions/50052421/mixed-models-with-two-random-effects-statsmodels
log_metrics = [i for i in metrics if 'log' in i and i not in ['log_Storm_Week', 'log_Population']]
for metric in log_metrics:
    formula = metric + ' ~ C(Campaign_Group) + C(Post) + log_Population+ C(Storm_Week) +   C(Campaign_Group):C(Post)'
    formula0 = metric + ' ~ C(Campaign_Group) + C(Post) + log_Population + C(Storm_Week) '

    vcf = {'re1': '0 + C(Group)', 're2': '0 + C(Week_Rel)'}
    alltime_wide['group'] = 1 # All in the same group

    mm = smf.mixedlm(formula, alltime_wide.dropna(subset=[metric]), groups = alltime_wide.dropna(subset=[metric])['group'],
                    vc_formula=vcf, re_formula="0")
    mm0 = smf.mixedlm(formula0, alltime_wide.dropna(subset=[metric]), groups = alltime_wide.dropna(subset=[metric])['group'],
                     vc_formula=vcf, re_formula="0")

    mm_fit = mm.fit(reml=False)
    mm_fit0 = mm0.fit(reml=False)

    p_val = 1 - chi2.cdf(-2*(mm_fit0.llf-mm_fit.llf), df=1)
    lr_stat = 2 * (
        mm_fit.llf - mm_fit0.llf
    )

    p_val = chi2.sf(
        lr_stat,
        df=2
    )

    trt_1_fitted_lift = np.exp(mm_fit.params['C(Campaign_Group)[T.Treated (Nighttime)]:C(Post)[T.1]']) - 1
    trt_1_fitted_lift_se = np.exp(mm_fit.bse['C(Campaign_Group)[T.Treated (Nighttime)]:C(Post)[T.1]']) - 1
    trt_1_fitted_lift_p = mm_fit.pvalues['C(Campaign_Group)[T.Treated (Nighttime)]:C(Post)[T.1]']
    trt_2_fitted_lift = np.exp(mm_fit.params['C(Campaign_Group)[T.Treated (Speeding)]:C(Post)[T.1]']) - 1
    trt_2_fitted_lift_se = np.exp(mm_fit.bse['C(Campaign_Group)[T.Treated (Speeding)]:C(Post)[T.1]']) - 1
    trt_2_fitted_lift_p = mm_fit.pvalues['C(Campaign_Group)[T.Treated (Speeding)]:C(Post)[T.1]']
    print('In {}, including ILS and non-ILS days {} to {}, treatment 1 saw a {}% (+/- {}%) impact relative to control. p-value: {} ' \
    .format(metric, 'pre_start_date', 'post_end_date', round(trt_1_fitted_lift * 100, 2), round(trt_1_fitted_lift_se * 100, 2), round(trt_1_fitted_lift_p, 4)))
    print('In {}, including ILS and non-ILS days {} to {}, treatment 2 saw a {}% (+/- {}%) impact relative to control. p-value: {} ' \
    .format(metric, 'pre_start_date', 'post_end_date', round(trt_2_fitted_lift * 100, 2), round(trt_2_fitted_lift_se * 100, 2), round(trt_2_fitted_lift_p, 4)))
    diagnostic_df = plot_mixed_diagnostics(
      mm_fit,
      alltime_wide.dropna(subset=[metric]),
      metric
    )
print(mm_fit.converged)
print(mm_fit.vcomp)
print(mm_fit.aic)

In [ ]:
# Refer to this: https://stackoverflow.com/questions/50052421/mixed-models-with-two-random-effects-statsmodels
for metric in log_metrics:
    formula = metric + ' ~ C(Campaign_Group) + C(Post) + log_Population + C(Storm_Week)+ temp_c + log_pct_age_18_24+ log_pct_age_65_plus+log_precipitation_mm +log_wind_speed_kph+   C(Campaign_Group):C(Post)'
    formula0 = metric + ' ~ C(Campaign_Group) + C(Post) + log_Population + C(Storm_Week)+ temp_c + log_pct_age_18_24+ log_pct_age_65_plus+log_precipitation_mm +log_wind_speed_kph'

    vcf = {'re1': '0 + C(Group)'}
    alltime_wide['group'] = 1 # All in the same group

    mm = smf.mixedlm(formula, alltime_wide.dropna(subset=[metric]), groups = alltime_wide.dropna(subset=[metric])['group'],
                    vc_formula=vcf, re_formula="0")
    mm0 = smf.mixedlm(formula0, alltime_wide.dropna(subset=[metric]), groups = alltime_wide.dropna(subset=[metric])['group'],
                     vc_formula=vcf, re_formula="0")

    mm_fit = mm.fit(reml=False)
    mm_fit0 = mm0.fit(reml=False)

    p_val = 1 - chi2.cdf(-2*(mm_fit0.llf-mm_fit.llf), df=1)
    lr_stat = 2 * (
        mm_fit.llf - mm_fit0.llf
    )

    p_val = chi2.sf(
        lr_stat,
        df=2
    )

    trt_1_fitted_lift = np.exp(mm_fit.params['C(Campaign_Group)[T.Treated (Nighttime)]:C(Post)[T.1]']) - 1
    trt_1_fitted_lift_se = np.exp(mm_fit.bse['C(Campaign_Group)[T.Treated (Nighttime)]:C(Post)[T.1]']) - 1
    trt_1_fitted_lift_p = mm_fit.pvalues['C(Campaign_Group)[T.Treated (Nighttime)]:C(Post)[T.1]']
    trt_2_fitted_lift = np.exp(mm_fit.params['C(Campaign_Group)[T.Treated (Speeding)]:C(Post)[T.1]']) - 1
    trt_2_fitted_lift_se = np.exp(mm_fit.bse['C(Campaign_Group)[T.Treated (Speeding)]:C(Post)[T.1]']) - 1
    trt_2_fitted_lift_p = mm_fit.pvalues['C(Campaign_Group)[T.Treated (Speeding)]:C(Post)[T.1]']
    print('In {}, including ILS and non-ILS days {} to {}, treatment 1 saw a {}% (+/- {}%) impact relative to control. p-value: {} ' \
    .format(metric, 'pre_start_date', 'post_end_date', round(trt_1_fitted_lift * 100, 2), round(trt_1_fitted_lift_se * 100, 2), round(trt_1_fitted_lift_p, 4)))
    print('In {}, including ILS and non-ILS days {} to {}, treatment 2 saw a {}% (+/- {}%) impact relative to control. p-value: {} ' \
    .format(metric, 'pre_start_date', 'post_end_date', round(trt_2_fitted_lift * 100, 2), round(trt_2_fitted_lift_se * 100, 2), round(trt_2_fitted_lift_p, 4)))
    diagnostic_df = plot_mixed_diagnostics(
      mm_fit,
      alltime_wide.dropna(subset=[metric]),
      metric
    )
print(mm_fit.converged)
print(mm_fit.vcomp)
print(mm_fit.aic)

In [ ]:
import statsmodels.formula.api as smf
from scipy.stats import chi2
# Refer to this: https://stackoverflow.com/questions/50052421/mixed-models-with-two-random-effects-statsmodels
log_metrics = [i for i in metrics if 'log' in i and i not in ['log_Storm_Week', 'log_Population']]
for metric in log_metrics:
    formula = metric + ' ~ C(Campaign_Group) + C(Post) + log_Population+ C(Storm_Week) +  C(Campaign_Group):C(Post)'
    formula0 = metric + ' ~ C(Campaign_Group) + C(Post) + log_Population + C(Storm_Week) '

    vcf = {'re1': '0 + C(Group)'}
    alltime_wide['group'] = 1 # All in the same group

    mm = smf.mixedlm(formula, alltime_wide.dropna(subset=[metric]), groups = alltime_wide.dropna(subset=[metric])['group'],
                    vc_formula=vcf, re_formula="0")
    mm0 = smf.mixedlm(formula0, alltime_wide.dropna(subset=[metric]), groups = alltime_wide.dropna(subset=[metric])['group'],
                     vc_formula=vcf, re_formula="0")

    mm_fit = mm.fit(reml=False)
    mm_fit0 = mm0.fit(reml=False)

    p_val = 1 - chi2.cdf(-2*(mm_fit0.llf-mm_fit.llf), df=1)
    lr_stat = 2 * (
        mm_fit.llf - mm_fit0.llf
    )

    p_val = chi2.sf(
        lr_stat,
        df=1
    )

    trt_1_fitted_lift = np.exp(mm_fit.params['C(Campaign_Group)[T.Treated (Nighttime)]:C(Post)[T.1]']) - 1
    trt_1_fitted_lift_se = np.exp(mm_fit.bse['C(Campaign_Group)[T.Treated (Nighttime)]:C(Post)[T.1]']) - 1
    trt_1_fitted_lift_p = mm_fit.pvalues['C(Campaign_Group)[T.Treated (Nighttime)]:C(Post)[T.1]']
    trt_2_fitted_lift = np.exp(mm_fit.params['C(Campaign_Group)[T.Treated (Speeding)]:C(Post)[T.1]']) - 1
    trt_2_fitted_lift_se = np.exp(mm_fit.bse['C(Campaign_Group)[T.Treated (Speeding)]:C(Post)[T.1]']) - 1
    trt_2_fitted_lift_p = mm_fit.pvalues['C(Campaign_Group)[T.Treated (Speeding)]:C(Post)[T.1]']
    print('In {}, {} to {}, treatment 1 saw a {}% (+/- {}%) impact relative to control. p-value: {} ' \
    .format(metric, 'pre_start_date', 'post_end_date', round(trt_1_fitted_lift * 100, 2), round(trt_1_fitted_lift_se * 100, 2), round(trt_1_fitted_lift_p, 4)))
    print('In {},  {} to {}, treatment 2 saw a {}% (+/- {}%) impact relative to control. p-value: {} ' \
    .format(metric, 'pre_start_date', 'post_end_date', round(trt_2_fitted_lift * 100, 2), round(trt_2_fitted_lift_se * 100, 2), round(trt_2_fitted_lift_p, 4)))
    diagnostic_df = plot_mixed_diagnostics(
      mm_fit,
      alltime_wide.dropna(subset=[metric]),
      metric
    )

In [ ]:
log_metrics = [
    i for i in metrics
    if "log" in i
    and i not in ["log_Storm_Week", "log_Population"]
]

aic_results = []

for metric in log_metrics:

    formula = (
        metric
        + " ~ C(Campaign_Group)"
        + " + C(Post)"
        + " + log_Population"
        + " + C(Storm_Week)"
        + " + Week_Rel"
        + " + C(Campaign_Group):C(Post)"
    )

    model_df = alltime_wide.dropna(
        subset=[
            metric,
            "Campaign_Group",
            "Post",
            "log_Population",
            "Storm_Week",
            "Group",
            "County",
            "Week_Rel"
        ]
    ).copy()

    model_df["group"] = 1

    random_effect_models = {
        "Group": {
            "Group": "0 + C(Group)"
        },
        "Group + County": {
            "Group": "0 + C(Group)",
            "County": "0 + C(County)"
        },
        "County": {
            "County": "0 + C(County)"
        },
        "Group + Week_Rel": {
            "Group": "0 + C(Group)",
            "Week_Rel": "0 + C(Week_Rel)"
        }
    }

    for model_name, vcf in random_effect_models.items():

        mm = smf.mixedlm(
            formula,
            model_df,
            groups=model_df["group"],
            vc_formula=vcf,
            re_formula="0"
        )

        mm_fit = mm.fit(reml=False)

        aic_results.append({
            "Metric": metric,
            "Model": model_name,
            "AIC": mm_fit.aic,
            "Converged": mm_fit.converged
        })
    # Ordinary DiD model
    did_fit = smf.ols(
        formula,
        data=model_df
    ).fit()

    aic_results.append({
        "Metric": metric,
        "Model": "OLS DiD",
        "AIC": did_fit.aic,
        "Converged": True
    })



aic_results = pd.DataFrame(aic_results)

aic_results["Delta_AIC"] = (
    aic_results["AIC"]
    - aic_results.groupby("Metric")["AIC"].transform("min")
)

display(
    aic_results
    .sort_values(["Metric", "Delta_AIC"])
    .round({"AIC": 2, "Delta_AIC": 2})
)

For every metric, ordinary OLS DiD had a substantially higher AIC than both the Group and County random-intercept models. This indicates that the independent-error OLS specification does not adequately capture the dependence among repeated observations. The lower AIC of the Group model suggests that observations belonging to the same randomized Group share unexplained variation, while the still lower AIC of the County model suggests that repeated measurements from the same county have particularly strong similarities across weeks. Allowing Groups or counties to have persistent above- or below-average baselines therefore describes the data better than treating every county-week observation as independent. Among the mixed-effects specifications, the Group-only model produced marginal and conditional residual diagnostics that most closely satisfied the model assumptions. Because treatment was also randomized at the Group level, I would move forward with the Group random-intercept mixed-effects DiD

In [ ]:
# @title
from scipy import stats

def plot_ols_diagnostics(
    did_model,
    model_df,
    metric
):

    # Ordinary fit for residual and influence diagnostics
    diagnostic_fit = did_model.fit()
    influence = diagnostic_fit.get_influence()

    diagnostic_df = model_df.copy().reset_index(drop=True)

    diagnostic_df["Fitted"] = np.asarray(
        diagnostic_fit.fittedvalues
    )

    diagnostic_df["Residual"] = np.asarray(
        diagnostic_fit.resid
    )

    diagnostic_df["Standardized_Residual"] = (
        influence.resid_studentized_internal
    )

    cooks_distance = influence.cooks_distance[0]

    fig, axs = plt.subplots(
        nrows=2,
        ncols=3,
        figsize=(18, 10)
    )

    # 1. Residuals versus fitted values
    sns.regplot(
        data=diagnostic_df,
        x="Fitted",
        y="Residual",
        lowess=True,
        scatter_kws={"alpha": 0.25},
        line_kws={"color": "red"},
        ax=axs[0, 0]
    )

    axs[0, 0].axhline(0, color="black", linestyle="--")
    axs[0, 0].set_title("Residuals versus fitted values")

    # 2. Normal Q-Q plot
    stats.probplot(
        diagnostic_df["Standardized_Residual"],
        dist="norm",
        plot=axs[0, 1]
    )

    axs[0, 1].set_title("Normal Q-Q plot")

    # 3. Scale-location plot
    diagnostic_df["Sqrt_Absolute_Residual"] = np.sqrt(
        np.abs(diagnostic_df["Standardized_Residual"])
    )

    sns.regplot(
        data=diagnostic_df,
        x="Fitted",
        y="Sqrt_Absolute_Residual",
        lowess=True,
        scatter_kws={"alpha": 0.25},
        line_kws={"color": "red"},
        ax=axs[0, 2]
    )

    axs[0, 2].set_title("Scale-location plot")
    axs[0, 2].set_ylabel("√|Standardized residual|")

    # 4. Residuals by randomized Group
    sns.boxplot(
        data=diagnostic_df,
        x="Group",
        y="Residual",
        showfliers=False,
        ax=axs[1, 0]
    )

    axs[1, 0].axhline(0, color="black", linestyle="--")
    axs[1, 0].set_title("Residuals by Group")
    axs[1, 0].tick_params(axis="x", rotation=45)

    # 5. Mean residual by relative week
    week_residuals = (
        diagnostic_df
        .groupby("Week_Rel", as_index=False)["Residual"]
        .mean()
        .sort_values("Week_Rel")
    )

    sns.lineplot(
        data=week_residuals,
        x="Week_Rel",
        y="Residual",
        marker="o",
        ax=axs[1, 1]
    )

    axs[1, 1].axhline(0, color="black", linestyle="--")
    axs[1, 1].set_title("Mean residual by relative week")

    # 6. Cook's distance
    axs[1, 2].scatter(
        np.arange(len(cooks_distance)),
        cooks_distance,
        alpha=0.3
    )

    axs[1, 2].axhline(
        4 / len(cooks_distance),
        color="red",
        linestyle="--",
        label="4/n"
    )

    axs[1, 2].set_title("Cook's distance")
    axs[1, 2].set_xlabel("Observation")
    axs[1, 2].legend()

    fig.suptitle(
        f"OLS DiD diagnostics: {metric}",
        fontsize=15
    )

    plt.tight_layout()
    plt.show()

    return diagnostic_df

In [ ]:
for metric in log_metrics:
    print(metric, 'metric')
    model_df = alltime_wide.dropna(
        subset=[
            metric,
            "Campaign_Group",
            "Post",
            "Group",
            "log_Population",
            "Storm_Week"
        ]
    ).copy()

    formula = (
        metric
        + " ~ C(Campaign_Group)"
        + " + C(Post)"
        + " + log_Population"
        + " + C(Storm_Week)"
        + " + C(Campaign_Group):C(Post)"
    )

    # Difference-in-differences model
    did_model = smf.ols(
        formula,
        data=model_df
    )

    # Cluster standard errors at randomized Group level
    did_fit = did_model.fit(
        cov_type="cluster",
        cov_kwds={
            "groups": model_df["Group"],
            "use_correction": True,
            "df_correction": True
        },
        use_t=True
    )


    nighttime_term = (
        "C(Campaign_Group)[T.Treated (Nighttime)]:"
        "C(Post)[T.1]"
    )

    speeding_term = (
        "C(Campaign_Group)[T.Treated (Speeding)]:"
        "C(Post)[T.1]"
    )

    for treatment, term in {
        "Nighttime": nighttime_term,
        "Speeding": speeding_term
    }.items():

        beta = did_fit.params[term]
        lower, upper = did_fit.conf_int().loc[term]

        effect = 100 * np.expm1(beta)
        ci_lower = 100 * np.expm1(lower)
        ci_upper = 100 * np.expm1(upper)
        p_value = did_fit.pvalues[term]

        print(
            f"{metric}, {treatment}: "
            f"{effect:.2f}% "
            f"(95% CI: {ci_lower:.2f}% to {ci_upper:.2f}%), "
            f"p-value: {p_value:.4f}"
        )
    diagnostic_df = plot_ols_diagnostics(
        did_model,
        model_df,
        metric
    )

In [ ]:
# @title
for metric in log_metrics:
  plot_df = (
      alltime_wide.groupby(
          ["Group", "Campaign_Group", "Post"],
          as_index=False
      )[metric]
      .mean()
  )

  plot_df["Period"] = plot_df["Post"].map({
      0: "Pre",
      1: "Post"
  })

  plt.figure(figsize=(10, 6))

  sns.pointplot(
      data=plot_df,
      x="Period",
      y=metric,
      hue="Campaign_Group",
      order=["Pre", "Post"],
      errorbar=("ci", 95),
      dodge=0.05,
      markers=["o", "s", "^"],
      linestyles=["-", "-", "-"]
  )

  plt.title(f"{dataset_name}: DiD Plot for {metric}")
  plt.xlabel("")
  plt.ylabel(metric)
  plt.legend(title="Campaign Group")
  plt.tight_layout()
  plt.show()

In [ ]:
additional_covariates = [
    "temp_c",
    "log_pct_age_18_24",
    "log_pct_age_65_plus",
    "log_precipitation_mm",
    "log_wind_speed_kph"
]


excluded_log_columns = [
    "log_Storm_Week",
    "log_Population",
    "log_temp_c",
    "log_median_household_income",
    "log_pct_age_18_24",
    "log_pct_age_65_plus",
    "log_precipitation_mm",
    "log_wind_speed_kph"
]


log_metrics = [
    metric
    for metric in metrics
    if metric.startswith("log_")
    and metric not in excluded_log_columns
]


# Original covariate specification
original_rhs = (
    "C(Campaign_Group)"
    " + C(Post)"
    " + log_Population"
    " + C(Storm_Week)"
    " + Week_Rel"
    " + C(Campaign_Group):C(Post)"
)


# Expanded covariate specification
expanded_rhs = (
    original_rhs
    + " + temp_c"
    + " + log_pct_age_18_24"
    + " + log_pct_age_65_plus"
    + " + log_precipitation_mm"
    + " + log_wind_speed_kph"
)


covariate_sets = {
    "Original": original_rhs,
    "Expanded": expanded_rhs
}


random_effect_models = {
    "Group": {
        "Group": "0 + C(Group)"
    },

    "Group + County": {
        "Group": "0 + C(Group)",
        "County": "0 + C(County)"
    },

    "County": {
        "County": "0 + C(County)"
    },

    "Group + Week_Rel": {
        "Group": "0 + C(Group)",
        "Week_Rel": "0 + C(Week_Rel)"
    }
}


results = []


for metric in log_metrics:

    # Use the same observations for every model
    model_df = (
        alltime_wide
        .replace([np.inf, -np.inf], np.nan)
        .dropna(
            subset=[
                metric,
                "Campaign_Group",
                "Post",
                "log_Population",
                "Storm_Week",
                "Week_Rel",
                "Group",
                "County"
            ]
            + additional_covariates
        )
        .copy()
    )

    model_df["group"] = 1

    for covariate_set, rhs in covariate_sets.items():

        formula = metric + " ~ " + rhs

        for model_name, vcf in random_effect_models.items():

            mm_fit = smf.mixedlm(
                formula,
                model_df,
                groups=model_df["group"],
                vc_formula=vcf,
                re_formula="0"
            ).fit(
                reml=False
            )

            k = (
                mm_fit.aic
                + 2 * mm_fit.llf
            ) / 2

            caic = (
                -2 * mm_fit.llf
                + k * (np.log(mm_fit.nobs) + 1)
            )

            results.append({
                "Metric": metric,
                "Model": model_name,
                "Covariate_Set": covariate_set,
                "AIC": mm_fit.aic,
                "BIC": mm_fit.bic,
                "CAIC": caic,
                "Converged": mm_fit.converged
            })


        # OLS DiD
        did_fit = smf.ols(
            formula,
            data=model_df
        ).fit()

        k = (
            did_fit.aic
            + 2 * did_fit.llf
        ) / 2

        caic = (
            -2 * did_fit.llf
            + k * (np.log(did_fit.nobs) + 1)
        )

        results.append({
            "Metric": metric,
            "Model": "OLS DiD",
            "Covariate_Set": covariate_set,
            "AIC": did_fit.aic,
            "BIC": did_fit.bic,
            "CAIC": caic,
            "Converged": True
        })


results = pd.DataFrame(results)


# Put Original and Expanded beside each other
covariate_comparison = results.pivot(
    index=["Metric", "Model"],
    columns="Covariate_Set",
    values=[
        "AIC",
        "BIC",
        "CAIC",
        "Converged"
    ]
)

covariate_comparison.columns = [
    f"{criterion}_{covariate_set}"
    for criterion, covariate_set
    in covariate_comparison.columns
]

covariate_comparison = (
    covariate_comparison
    .reset_index()
)


# Expanded minus Original
for criterion in ["AIC", "BIC", "CAIC"]:

    covariate_comparison[
        f"{criterion}_Change"
    ] = (
        covariate_comparison[
            f"{criterion}_Expanded"
        ]
        - covariate_comparison[
            f"{criterion}_Original"
        ]
    )


display(
    covariate_comparison
    .sort_values(
        ["Metric", "AIC_Change"]
    )
    .round(2)
)

In [ ]:
# Refer to this: https://stackoverflow.com/questions/50052421/mixed-models-with-two-random-effects-statsmodels
for metric in log_metrics:
    formula = metric + ' ~ C(Campaign_Group) + C(Post) + log_Population + C(Storm_Week)+ temp_c + log_pct_age_18_24+ log_pct_age_65_plus+log_precipitation_mm +log_wind_speed_kph+   C(Campaign_Group):C(Post)'
    formula0 = metric + ' ~ C(Campaign_Group) + C(Post) + log_Population + C(Storm_Week)+ temp_c + log_pct_age_18_24+ log_pct_age_65_plus+log_precipitation_mm +log_wind_speed_kph'

    vcf = {'re1': '0 + C(Group)'}
    nighttime_wide['group'] = 1 # All in the same group

    mm = smf.mixedlm(formula, nighttime_wide.dropna(subset=[metric]), groups = nighttime_wide.dropna(subset=[metric])['group'],
                    vc_formula=vcf, re_formula="0")
    mm0 = smf.mixedlm(formula0, nighttime_wide.dropna(subset=[metric]), groups = nighttime_wide.dropna(subset=[metric])['group'],
                     vc_formula=vcf, re_formula="0")

    mm_fit = mm.fit(reml=False)
    mm_fit0 = mm0.fit(reml=False)

    p_val = 1 - chi2.cdf(-2*(mm_fit0.llf-mm_fit.llf), df=1)
    lr_stat = 2 * (
        mm_fit.llf - mm_fit0.llf
    )

    p_val = chi2.sf(
        lr_stat,
        df=2
    )

    trt_1_fitted_lift = np.exp(mm_fit.params['C(Campaign_Group)[T.Treated (Nighttime)]:C(Post)[T.1]']) - 1
    trt_1_fitted_lift_se = np.exp(mm_fit.bse['C(Campaign_Group)[T.Treated (Nighttime)]:C(Post)[T.1]']) - 1
    trt_1_fitted_lift_p = mm_fit.pvalues['C(Campaign_Group)[T.Treated (Nighttime)]:C(Post)[T.1]']
    trt_2_fitted_lift = np.exp(mm_fit.params['C(Campaign_Group)[T.Treated (Speeding)]:C(Post)[T.1]']) - 1
    trt_2_fitted_lift_se = np.exp(mm_fit.bse['C(Campaign_Group)[T.Treated (Speeding)]:C(Post)[T.1]']) - 1
    trt_2_fitted_lift_p = mm_fit.pvalues['C(Campaign_Group)[T.Treated (Speeding)]:C(Post)[T.1]']
    print('In {},  {} to {}, treatment 1 saw a {}% (+/- {}%) impact relative to control. p-value: {} ' \
    .format(metric, 'pre_start_date', 'post_end_date', round(trt_1_fitted_lift * 100, 2), round(trt_1_fitted_lift_se * 100, 2), round(trt_1_fitted_lift_p, 4)))
    print('In {},  {} to {}, treatment 2 saw a {}% (+/- {}%) impact relative to control. p-value: {} ' \
    .format(metric, 'pre_start_date', 'post_end_date', round(trt_2_fitted_lift * 100, 2), round(trt_2_fitted_lift_se * 100, 2), round(trt_2_fitted_lift_p, 4)))
    raw_pvalues = [
        trt_1_fitted_lift_p,
        trt_2_fitted_lift_p
    ]

    bonferroni_pvalues = multipletests(
        raw_pvalues,
        method="bonferroni"
    )[1]

    holm_pvalues = multipletests(
        raw_pvalues,
        method="holm"
    )[1]
    print(
    f"In {metric}, treatment 1 saw a "
    f"{trt_1_fitted_lift * 100:.2f}% "
    f"(+/- {trt_1_fitted_lift_se * 100:.2f}%) impact. "
    f"Raw p-value: {trt_1_fitted_lift_p:.4f}, "
    f"Bonferroni p-value: {bonferroni_pvalues[0]:.4f}, "
    f"Holm p-value: {holm_pvalues[0]:.4f}"
    )

    print(
        f"In {metric}, treatment 2 saw a "
        f"{trt_2_fitted_lift * 100:.2f}% "
        f"(+/- {trt_2_fitted_lift_se * 100:.2f}%) impact. "
        f"Raw p-value: {trt_2_fitted_lift_p:.4f}, "
        f"Bonferroni p-value: {bonferroni_pvalues[1]:.4f}, "
        f"Holm p-value: {holm_pvalues[1]:.4f}"
    )
    diagnostic_df = plot_mixed_diagnostics(
      mm_fit,
      nighttime_wide.dropna(subset=[metric]),
      metric
    )


In [ ]:
# Refer to this: https://stackoverflow.com/questions/50052421/mixed-models-with-two-random-effects-statsmodels
from statsmodels.stats.multitest import multipletests


for metric in log_metrics:
    formula = metric + ' ~ C(Campaign_Group) + C(Post) + log_Population + C(Storm_Week)+ temp_c + log_pct_age_18_24+ log_pct_age_65_plus+log_precipitation_mm +log_wind_speed_kph+   C(Campaign_Group):C(Post)'
    formula0 = metric + ' ~ C(Campaign_Group) + C(Post) + log_Population + C(Storm_Week)+ temp_c + log_pct_age_18_24+ log_pct_age_65_plus+log_precipitation_mm +log_wind_speed_kph'

    vcf = {'re1': '0 + C(Group)'}
    alltime_wide['group'] = 1 # All in the same group

    mm = smf.mixedlm(formula, alltime_wide.dropna(subset=[metric]), groups = alltime_wide.dropna(subset=[metric])['group'],
                    vc_formula=vcf, re_formula="0")
    mm0 = smf.mixedlm(formula0, alltime_wide.dropna(subset=[metric]), groups = alltime_wide.dropna(subset=[metric])['group'],
                     vc_formula=vcf, re_formula="0")

    mm_fit = mm.fit(reml=False)
    mm_fit0 = mm0.fit(reml=False)

    p_val = 1 - chi2.cdf(-2*(mm_fit0.llf-mm_fit.llf), df=1)
    lr_stat = 2 * (
        mm_fit.llf - mm_fit0.llf
    )

    p_val = chi2.sf(
        lr_stat,
        df=2
    )

    trt_1_fitted_lift = np.exp(mm_fit.params['C(Campaign_Group)[T.Treated (Nighttime)]:C(Post)[T.1]']) - 1
    trt_1_fitted_lift_se = np.exp(mm_fit.bse['C(Campaign_Group)[T.Treated (Nighttime)]:C(Post)[T.1]']) - 1
    trt_1_fitted_lift_p = mm_fit.pvalues['C(Campaign_Group)[T.Treated (Nighttime)]:C(Post)[T.1]']
    trt_2_fitted_lift = np.exp(mm_fit.params['C(Campaign_Group)[T.Treated (Speeding)]:C(Post)[T.1]']) - 1
    trt_2_fitted_lift_se = np.exp(mm_fit.bse['C(Campaign_Group)[T.Treated (Speeding)]:C(Post)[T.1]']) - 1
    trt_2_fitted_lift_p = mm_fit.pvalues['C(Campaign_Group)[T.Treated (Speeding)]:C(Post)[T.1]']
    raw_pvalues = [
        trt_1_fitted_lift_p,
        trt_2_fitted_lift_p
    ]

    bonferroni_pvalues = multipletests(
        raw_pvalues,
        method="bonferroni"
    )[1]

    holm_pvalues = multipletests(
        raw_pvalues,
        method="holm"
    )[1]
    print('In {},  {} to {}, treatment 1 saw a {}% (+/- {}%) impact relative to control. p-value: {} ' \
    .format(metric, 'pre_start_date', 'post_end_date', round(trt_1_fitted_lift * 100, 2), round(trt_1_fitted_lift_se * 100, 2), round(trt_1_fitted_lift_p, 4)))
    print('In {},  {} to {}, treatment 2 saw a {}% (+/- {}%) impact relative to control. p-value: {} ' \
    .format(metric, 'pre_start_date', 'post_end_date', round(trt_2_fitted_lift * 100, 2), round(trt_2_fitted_lift_se * 100, 2), round(trt_2_fitted_lift_p, 4)))
    print(
    f"In {metric}, treatment 1 saw a "
    f"{trt_1_fitted_lift * 100:.2f}% "
    f"(+/- {trt_1_fitted_lift_se * 100:.2f}%) impact. "
    f"Raw p-value: {trt_1_fitted_lift_p:.4f}, "
    f"Bonferroni p-value: {bonferroni_pvalues[0]:.4f}, "
    f"Holm p-value: {holm_pvalues[0]:.4f}"
    )

    print(
        f"In {metric}, treatment 2 saw a "
        f"{trt_2_fitted_lift * 100:.2f}% "
        f"(+/- {trt_2_fitted_lift_se * 100:.2f}%) impact. "
        f"Raw p-value: {trt_2_fitted_lift_p:.4f}, "
        f"Bonferroni p-value: {bonferroni_pvalues[1]:.4f}, "
        f"Holm p-value: {holm_pvalues[1]:.4f}"
    )
    diagnostic_df = plot_mixed_diagnostics(
      mm_fit,
      alltime_wide.dropna(subset=[metric]),
      metric
    )


In [ ]:
did_results = []

terms = {
    "Nighttime": (
        "C(Campaign_Group)[T.Treated (Nighttime)]:C(Post)[T.1]"
    ),
    "Speeding": (
        "C(Campaign_Group)[T.Treated (Speeding)]:C(Post)[T.1]"
    )
}

for metric in log_metrics:

    # Same observations for both specifications
    model_df = (
        alltime_wide
        .replace([np.inf, -np.inf], np.nan)
        .dropna(
            subset=[
                metric,
                "Campaign_Group",
                "Post",
                "Group",
                "log_Population",
                "Storm_Week",
                "Week_Rel"
            ] + additional_covariates
        )
    )

    for covariate_set, rhs in covariate_sets.items():

        did_fit = smf.ols(
            metric + " ~ " + rhs,
            data=model_df
        ).fit(
            cov_type="cluster",
            cov_kwds={"groups": model_df["Group"]},
            use_t=True
        )

        for treatment, term in terms.items():

            lower, upper = did_fit.conf_int().loc[term]

            did_results.append({
                "Metric": metric,
                "Covariates": covariate_set,
                "Treatment": treatment,
                "Effect (%)": 100 * np.expm1(did_fit.params[term]),
                "CI Lower (%)": 100 * np.expm1(lower),
                "CI Upper (%)": 100 * np.expm1(upper),
                "Raw p-value": did_fit.pvalues[term]
            })

did_results = pd.DataFrame(did_results)

display(did_results.round(4))